# MediGuide — Comparative ML Baseline (Mentor Demo)
**Notebook:** `notebooks/02_mentor_demo_model.ipynb` &nbsp;|&nbsp; **Dataset:** UCI Heart Disease (Cleveland) &nbsp;|&nbsp; **Runtime:** Google Colab, top-to-bottom, no uploads needed

---

> ## ⚠️ Safety Notice — Please Read
> **This ML model is a _comparative demonstration_ only.** It does **not** diagnose, and it does **not** supersede or override the rule-based emergency triage system.
> In MediGuide, the deterministic triage engine evaluates emergency **red flags FIRST**. If any red flag fires, the user is directed to emergency care **regardless of what this model predicts**.
> The model output is shown to users only as a supplementary **"Comparative ML Estimate"** next to the rule-based concern score, using wording such as *"possible cardiovascular concern"* and *"consult a healthcare professional"*.

## 1. Role of this model inside MediGuide

```
User symptoms + health info
        │
        ▼
┌──────────────────────────────┐
│ 1. Rule-based red-flag triage │──► RED FLAG? ──► EMERGENCY CARE (always overrides ML)
└──────────────────────────────┘
        │ no red flag
        ▼
┌──────────────────────────────┐      ┌─────────────────────────────────┐
│ 2. Rule-based concern score   │      │ 3. Comparative ML Estimate       │
│    (primary, explainable)     │      │    (this notebook; supplementary)│
└──────────────────────────────┘      └─────────────────────────────────┘
        └──────────────┬─────────────────────────┘
                       ▼
     Care recommendation: self-monitor / routine checkup / prompt consultation / emergency care
```

**What this notebook demonstrates**
1. Loading and cleaning the Cleveland data (missing values, binarised target).
2. Leakage-safe preprocessing (scaler fitted on training data only; scaling inside each CV fold).
3. Three baseline models compared with cross-validation and a held-out test set.
4. Sensitivity-focused evaluation (false negatives matter most in screening).
5. Explainability through feature importances.
6. A reusable `predict_patient_risk()` function — the prototype of `src/models/predict.py`.

### 1.1 Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})

print("Setup complete.")

### 1.2 Fetch the dataset directly from UCI
The file has no header row, so column names are supplied manually. Missing values are encoded as `?` (only in `ca` and `thal`), which makes pandas read those two columns as text — we deal with that in Section 2.

In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
FEATURES = COLUMNS[:-1]   # the 13 predictors

df_raw = pd.read_csv(URL, header=None, names=COLUMNS)
print(f"Loaded {df_raw.shape[0]} patients x {df_raw.shape[1]} columns")
display(df_raw.head())

### 1.3 Feature dictionary
| Feature | Meaning | Encoding |
|---|---|---|
| `age` | Age in years | numeric |
| `sex` | Sex | 1 = male, 0 = female |
| `cp` | Chest pain type | 1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic |
| `trestbps` | Resting blood pressure (mm Hg) | numeric |
| `chol` | Serum cholesterol (mg/dl) | numeric |
| `fbs` | Fasting blood sugar > 120 mg/dl | 1 = true, 0 = false |
| `restecg` | Resting ECG result | 0 normal, 1 ST-T abnormality, 2 LV hypertrophy |
| `thalach` | Maximum heart rate achieved | numeric |
| `exang` | Exercise-induced angina | 1 = yes, 0 = no |
| `oldpeak` | ST depression induced by exercise vs rest | numeric |
| `slope` | Slope of peak-exercise ST segment | 1 upsloping, 2 flat, 3 downsloping |
| `ca` | Major vessels (0–3) coloured by fluoroscopy | 0–3 |
| `thal` | Thalassemia / thallium stress-test result | 3 normal, 6 fixed defect, 7 reversible defect |
| `num` | **Target**: angiographic disease status | 0 = healthy, 1–4 = increasing disease stages |

## 2. Data Cleaning & Preprocessing
Steps:
1. Count the `?` placeholders.
2. Coerce all columns to numeric (`?` → `NaN`).
3. Impute missing values with the **median** (robust to outliers).
4. Binarise the target: `0 → 0 (no disease)`, `1–4 → 1 (disease present)`.

> **Note on imputation order:** only a handful of values are missing, so imputing before the split has a negligible effect here. In the production pipeline (`src/models/`), the imputer will be fitted on training data only, inside a `Pipeline`.

In [ ]:
# 1) Count '?' placeholders in the raw data
q_counts = (df_raw == "?").sum()
print("'?' placeholders per column:")
print(q_counts[q_counts > 0].to_string(), "\n")

# 2) Coerce to numeric -> '?' becomes NaN
df = df_raw.apply(pd.to_numeric, errors="coerce")
print("Missing values after coercion:")
print(df.isna().sum()[df.isna().sum() > 0].to_string(), "\n")

# 3) Median imputation
medians = df[FEATURES].median()
df[FEATURES] = df[FEATURES].fillna(medians)
print("Medians used for imputation (ca, thal):", medians[["ca", "thal"]].to_dict())
print("Missing values remaining:", int(df.isna().sum().sum()), "\n")

# 4) Binarise the target
df["target"] = (df["num"] > 0).astype(int)
summary = pd.DataFrame({
    "original num (0-4)": df["num"].value_counts().sort_index(),
})
display(summary)
print("Binary target counts:\n", df["target"].value_counts().rename({0: "0 - no disease", 1: "1 - disease"}).to_string())

## 3. Exploratory Data Analysis
Two quick checks before modelling: **class balance** (does accuracy mean anything?) and **feature correlations** (which signals relate to the target, and is there redundancy?).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.countplot(data=df, x="target", palette=["#4C9F70", "#D1495B"], ax=ax)
ax.set_xticklabels(["No disease (0)", "Disease (1)"])
ax.set_xlabel("Binary target")
ax.set_ylabel("Number of patients")
ax.set_title("Target Class Balance (Cleveland, n = %d)" % len(df))
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}  ({p.get_height() / len(df):.1%})",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, df["target"].value_counts().max() * 1.15)
plt.tight_layout()
plt.show()

The classes are reasonably balanced (roughly 54 % / 46 %), so plain accuracy is a meaningful metric and no resampling is needed. We still report recall and ROC-AUC because, for screening, **missing a sick patient is costlier than a false alarm**.

In [ ]:
corr = df[FEATURES + ["target"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, linewidths=0.5, cbar_kws={"label": "Pearson correlation"}, ax=ax)
ax.set_title("Correlation Heatmap - Clinical Features and Binary Target")
plt.tight_layout()
plt.show()

print("Correlation of each feature with the target (sorted by absolute strength):")
display(corr["target"].drop("target").sort_values(key=abs, ascending=False).to_frame("corr_with_target").round(3))

**Reading the heatmap:** `thal`, `ca`, `oldpeak`, `cp`, `exang` and `thalach` show the strongest associations with disease. No pair of predictors is extremely collinear, so all 13 features are kept. (Correlation is only a linear, univariate view; the models below can capture non-linear interactions.)

## 4. Data Splitting & Scaling
- **Stratified 80/20 split** keeps the disease ratio the same in train and test.
- **`StandardScaler` is fitted on the training set only**, then applied to the test set. Fitting on all data would leak test-set mean/variance into training.

In [ ]:
X = df[FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler().fit(X_train)            # fit on TRAIN only
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=FEATURES, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=FEATURES, index=X_test.index)

print(f"Train: {X_train.shape[0]} patients | Test: {X_test.shape[0]} patients")
print(f"Disease rate  - train: {y_train.mean():.1%} | test: {y_test.mean():.1%}")
print(f"Scaled train mean (should be ~0): {X_train_s.mean().abs().max():.2e}")
print(f"Scaled test  mean (not exactly 0, as expected): {X_test_s.mean().abs().max():.3f}")

## 5. Model Training & Cross-Validation
Three baselines of increasing complexity. Hyperparameters are deliberately conservative (shallow trees, modest ensembles) because the dataset is small (~240 training rows) and over-fitting is the main risk.

| Model | Why it is here |
|---|---|
| Logistic Regression | Interpretable linear baseline |
| Random Forest | Non-linear, robust, gives feature importances |
| Gradient Boosting | Strong tabular learner; sequentially corrects errors |

**5-fold stratified cross-validation** runs on the **training set only**. Each model is wrapped in a `Pipeline(StandardScaler → model)` so the scaler is re-fitted inside every fold, which keeps the validation folds leakage-free. The test set stays untouched until Section 6.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=2,
        random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", clone(model))])
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=["accuracy", "roc_auc"])
    cv_rows.append({
        "Model": name,
        "CV Accuracy (mean)": res["test_accuracy"].mean(),
        "CV Accuracy (std)": res["test_accuracy"].std(),
        "CV ROC-AUC (mean)": res["test_roc_auc"].mean(),
        "CV ROC-AUC (std)": res["test_roc_auc"].std(),
    })

cv_df = pd.DataFrame(cv_rows).set_index("Model")
print("5-fold stratified cross-validation (training set only)")
display(cv_df.round(3).style.background_gradient(cmap="Greens", subset=["CV Accuracy (mean)", "CV ROC-AUC (mean)"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, metric, color in zip(axes, ["Accuracy", "ROC-AUC"], ["#3E7CB1", "#81A4CD"]):
    ax.bar(cv_df.index, cv_df[f"CV {metric} (mean)"], yerr=cv_df[f"CV {metric} (std)"],
           capsize=6, color=color, edgecolor="black")
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel(metric)
    ax.set_title(f"5-Fold CV {metric} (mean +/- std)")
    ax.tick_params(axis="x", rotation=15)
    for i, v in enumerate(cv_df[f"CV {metric} (mean)"]):
        ax.text(i, v + 0.015, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

The standard deviations show fold-to-fold variability: with ~48 patients per validation fold, differences of a point or two between models are **not statistically meaningful**. All three models land in a similar range, which is typical for the Cleveland data.

## 6. Holdout Evaluation
Each model is now fitted on the full scaled training set and scored **once** on the untouched 20 % test set. Predictions use the default 0.5 probability threshold.

Metric reminder (positive class = disease):
- **Recall / sensitivity** — of the truly sick patients, how many did we catch? *(most important for screening)*
- **Precision** — of those flagged, how many were truly sick?
- **ROC-AUC** — threshold-independent ranking quality.

In [ ]:
fitted = {name: clone(m).fit(X_train_s, y_train) for name, m in models.items()}

results, probas, preds = [], {}, {}
for name, model in fitted.items():
    proba = model.predict_proba(X_test_s)[:, 1]
    pred = (proba >= 0.5).astype(int)
    probas[name], preds[name] = proba, pred
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall (Sensitivity)": recall_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "Specificity": tn / (tn + fp),
        "False Negatives": fn,
    })

metrics_df = pd.DataFrame(results).set_index("Model")
print(f"Holdout test set: n = {len(y_test)} ({int(y_test.sum())} disease, {int((1 - y_test).sum())} healthy)")
display(
    metrics_df.style
    .format({c: "{:.3f}" for c in metrics_df.columns if c != "False Negatives"})
    .highlight_max(subset=[c for c in metrics_df.columns if c != "False Negatives"], color="#c8e6c9")
    .highlight_min(subset=["False Negatives"], color="#c8e6c9")
)

> **Caveat:** the test set contains only 61 patients, so each metric moves several percentage points if a single patient is reclassified. Treat the table as an illustration of the mechanism, not as a validated clinical performance claim.

## 7. Visualizations — Confusion Matrices and ROC Curves
In a screening context, the **bottom-left cell (false negatives)** is the dangerous one: a sick patient labelled healthy. The panels below show how many each model produces.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (name, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Healthy (0)", "Disease (1)"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    fn = cm[1, 0]
    sens = recall_score(y_test, pred)
    ax.set_title(f"{name}\nFalse negatives: {fn} | Sensitivity: {sens:.1%}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.grid(False)
fig.suptitle("Confusion Matrices - Holdout Test Set (threshold = 0.5)", fontsize=15, fontweight="bold", y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
colors = {"Logistic Regression": "#1f77b4", "Random Forest": "#2ca02c", "Gradient Boosting": "#d62728"}

fig, ax = plt.subplots(figsize=(7.5, 6.5))
for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, lw=2.2, color=colors[name], label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random guess (AUC = 0.500)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate (1 - Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity / Recall)")
ax.set_title("ROC Curves - Holdout Test Set")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 7.1 Sensitivity vs. specificity at different alert thresholds
Because missing a sick patient is costlier than over-flagging, the screening threshold does not have to be 0.5. Lowering it raises sensitivity at the price of more false alarms. The final operating point should be chosen together with the clinical mentor and validated on more data.

In [ ]:
thr_rows = []
for name, proba in probas.items():
    for thr in [0.5, 0.4, 0.3]:
        pred = (proba >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        thr_rows.append({"Model": name, "Threshold": thr,
                         "Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp),
                         "False Negatives": fn, "False Positives": fp})
thr_df = pd.DataFrame(thr_rows)
display(thr_df.style.format({"Sensitivity": "{:.3f}", "Specificity": "{:.3f}", "Threshold": "{:.1f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 4.8))
for name in probas:
    sub = thr_df[thr_df["Model"] == name]
    ax.plot(sub["Threshold"], sub["Sensitivity"], marker="o", lw=2, color=colors[name], label=f"{name} - sensitivity")
ax.invert_xaxis()
ax.set_xticks([0.5, 0.4, 0.3])
ax.set_ylim(0.5, 1.02)
ax.set_xlabel("Decision threshold (lower = more cautious)")
ax.set_ylabel("Sensitivity")
ax.set_title("Sensitivity Gained by Lowering the Alert Threshold")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Explainability — Which Clinical Factors Drive the Predictions?
Tree ensembles expose **impurity-based feature importances**: how much each feature reduces classification error across all splits. This shows the mentor which inputs the model relies on, and lets us sanity-check them against clinical knowledge.

> **Caveats:** impurity importances show *association*, not causation, and can favour continuous/high-cardinality features. For a stricter analysis we can add permutation importance or SHAP in a later iteration.

In [ ]:
FEATURE_LABELS = {
    "age": "age", "sex": "sex", "cp": "cp (chest pain type)", "trestbps": "trestbps (resting BP)",
    "chol": "chol (cholesterol)", "fbs": "fbs (fasting sugar)", "restecg": "restecg (resting ECG)",
    "thalach": "thalach (max heart rate)", "exang": "exang (exercise angina)",
    "oldpeak": "oldpeak (ST depression)", "slope": "slope (ST slope)",
    "ca": "ca (vessels coloured)", "thal": "thal (thallium test)",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=False)
for ax, name, color in zip(axes, ["Random Forest", "Gradient Boosting"], ["#2ca02c", "#d62728"]):
    imp = pd.Series(fitted[name].feature_importances_, index=FEATURES).sort_values()
    ax.barh([FEATURE_LABELS[f] for f in imp.index], imp.values, color=color, edgecolor="black")
    ax.set_xlabel("Importance (share of total impurity reduction)")
    ax.set_title(f"Feature Importances - {name}")
    for i, v in enumerate(imp.values):
        ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
    ax.set_xlim(0, imp.max() * 1.18)
plt.tight_layout()
plt.show()

top_rf = pd.Series(fitted["Random Forest"].feature_importances_, index=FEATURES).sort_values(ascending=False).head(5)
print("Top-5 drivers (Random Forest):", ", ".join(f"{f} ({v:.2f})" for f, v in top_rf.items()))

**Interpretation for the mentor:** the dominant features are typically `thal`, `ca`, `cp`, `oldpeak`, `thalach` and `exang` (stress-test and angiography-related measurements plus chest-pain type), which matches established cardiology risk factors. 

**Integration caveat:** `thal` and `ca` come from clinical tests (thallium stress test, fluoroscopy) that a lay user filling in a symptom questionnaire will usually **not** have. When this model is wired into MediGuide, the plan must either (a) restrict the ML estimate to users who can supply these values, or (b) retrain on the subset of features obtainable from the questionnaire, and report the resulting performance drop honestly.

## 9. Interactive Mentor Demo — `predict_patient_risk()`
This is the prototype of what will live in `src/models/predict.py`. It takes a dictionary of the 13 features, applies the **training-fitted scaler**, queries the model and returns a probability, a binary class and a risk-level label.

Risk bands (illustrative; to be calibrated with the mentor): `< 0.35` Low · `0.35–0.65` Moderate · `≥ 0.65` High.

> ⚠️ In production this function is called **only after** the rule-based triage engine has found no emergency red flag, and its output is labelled *"Comparative ML Estimate"*.

In [ ]:
DEMO_MODEL_NAME = "Random Forest"      # change to "Logistic Regression" / "Gradient Boosting" to compare

def _risk_level(p: float) -> str:
    if p < 0.35:
        return "Low"
    if p < 0.65:
        return "Moderate"
    return "High"

def predict_patient_risk(patient_dict: dict, model_name: str = DEMO_MODEL_NAME, threshold: float = 0.5) -> dict:
    """Return the comparative ML estimate for one patient.

    patient_dict : dict with exactly the 13 feature keys (see feature dictionary).
    Returns      : dict with probability, binary class, risk level and a disclaimer.
    """
    missing = [f for f in FEATURES if f not in patient_dict]
    extra = [k for k in patient_dict if k not in FEATURES]
    if missing or extra:
        raise ValueError(f"Missing features: {missing} | Unexpected keys: {extra}")

    row = pd.DataFrame([patient_dict], columns=FEATURES).astype(float)
    row_scaled = pd.DataFrame(scaler.transform(row), columns=FEATURES)   # training-fitted scaler

    proba = float(fitted[model_name].predict_proba(row_scaled)[0, 1])
    return {
        "model": model_name,
        "probability_of_disease": round(proba, 3),
        "predicted_class": int(proba >= threshold),
        "risk_level": _risk_level(proba),
        "disclaimer": ("Comparative ML estimate only - not a diagnosis. "
                       "Emergency red flags are handled by the rule-based triage engine, which always takes priority."),
    }

def show_prediction(title: str, patient: dict, **kwargs):
    out = predict_patient_risk(patient, **kwargs)
    print("=" * 66)
    print(title)
    print("=" * 66)
    print("Input:", patient)
    print("-" * 66)
    print(f"Model                : {out['model']}")
    print(f"Predicted probability: {out['probability_of_disease']:.1%}")
    print(f"Predicted class      : {out['predicted_class']}  ({'disease pattern' if out['predicted_class'] else 'no disease pattern'})")
    print(f"Risk level           : {out['risk_level'].upper()}")
    print(f"Note                 : {out['disclaimer']}")
    print()
    return out

# ---- Sample execution: HIGH-RISK patient ------------------------------------
high_risk_patient = {
    "age": 63, "sex": 1, "cp": 4, "trestbps": 150, "chol": 280, "fbs": 1,
    "restecg": 2, "thalach": 110, "exang": 1, "oldpeak": 3.0, "slope": 2,
    "ca": 2, "thal": 7,
}
_ = show_prediction("SAMPLE 1 - High-risk profile", high_risk_patient)

# ---- Contrast case: LOW-RISK patient -----------------------------------------
low_risk_patient = {
    "age": 41, "sex": 0, "cp": 2, "trestbps": 118, "chol": 190, "fbs": 0,
    "restecg": 0, "thalach": 172, "exang": 0, "oldpeak": 0.0, "slope": 1,
    "ca": 0, "thal": 3,
}
_ = show_prediction("SAMPLE 2 - Low-risk profile (contrast)", low_risk_patient)

# ---- All three models on the high-risk patient --------------------------------
print("Same high-risk patient, all models:")
display(pd.DataFrame([predict_patient_risk(high_risk_patient, model_name=m) for m in fitted])
        [["model", "probability_of_disease", "predicted_class", "risk_level"]])

### 9.1 (Optional) Serialise the demo model
This mirrors the planned hand-off to the FastAPI backend: the fitted scaler, model and feature order are saved together so inference uses exactly the same preprocessing as training.

In [ ]:
import joblib

artifact = {"scaler": scaler, "model": fitted[DEMO_MODEL_NAME], "features": FEATURES, "model_name": DEMO_MODEL_NAME}
joblib.dump(artifact, "mediguide_baseline.joblib")
print("Saved mediguide_baseline.joblib (in the Colab working directory)")

## 10. Summary and Next Steps

**Key takeaways**
- All three baselines reach comparable cross-validated and holdout performance on Cleveland; the choice between them is not statistically decisive at this sample size.
- Lowering the alert threshold trades false alarms for higher sensitivity, which fits a screening setting.
- Feature importances agree with clinical expectations, giving a basic plausibility check.

**Limitations to state openly**
- Small, single-source dataset (297 patients, one clinic, collected decades ago); results may not generalise.
- Several predictors (`thal`, `ca`, `oldpeak`) require clinical testing and are not available from a symptom questionnaire.
- No probability calibration, external validation, or fairness analysis has been performed.

**Planned next steps**
1. Serialise the chosen model and load it in `src/models/predict.py`.
2. Expose it through FastAPI as a *Comparative ML Estimate* that supplements, and never overrides, the rule-based concern score.
3. Evaluate a reduced-feature model that uses only questionnaire-obtainable inputs.
4. Add calibration and permutation/SHAP explanations.
5. Extend the same modular pipeline to diabetes and respiratory screening.

> ⚠️ **Reminder:** this notebook is a demonstration of the comparative ML mechanism. It is **not** a diagnostic tool, and it never supersedes the rule-based emergency triage system.